# SVD Career Recommendation System

This notebook implements a career recommendation system using **SVD (Singular Value Decomposition)** for dimensionality reduction and cosine similarity.

## Step 1: Import Required Libraries

In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Libraries imported successfully!")

## Step 2: Load Career Dataset

In [ ]:
# Load careers dataset
with open("careers_dataset_50plus.json", "r") as f:
    careers_data = json.load(f)

print(f"✅ Loaded {len(careers_data)} careers from dataset")
print(f"\nSample career: {careers_data[0]['career_name']}")

## Step 3: Prepare Career Data

In [ ]:
def safe_list_to_string(data):
    """Convert list or string to clean string format"""
    if isinstance(data, list):
        return " ".join([str(item) for item in data if item])
    elif isinstance(data, str):
        return data
    else:
        return ""

# Create DataFrame
careers_df = pd.DataFrame(careers_data)

# Combine all text fields for each career
def combine_career_text(row):
    parts = []
    parts.append(str(row.get('career_name', '')))
    parts.append(str(row.get('category', '')))
    parts.append(safe_list_to_string(row.get('required_skills', [])))
    parts.append(safe_list_to_string(row.get('recommended_courses', [])))
    parts.append(safe_list_to_string(row.get('projects', [])))
    parts.append(str(row.get('roadmap', '')))
    return " ".join([p for p in parts if p])

careers_df['combined_text'] = careers_df.apply(combine_career_text, axis=1)

print(f"✅ Prepared {len(careers_df)} career profiles")
print(f"\nFirst career combined text preview:")
print(careers_df['combined_text'].iloc[0][:200] + "...")

## Step 4: Build TF-IDF Matrix

In [ ]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1
)

# Fit and transform career texts
tfidf_matrix = tfidf.fit_transform(careers_df['combined_text'])

print(f"✅ TF-IDF matrix created: {tfidf_matrix.shape}")
print(f"   - {tfidf_matrix.shape[0]} careers")
print(f"   - {tfidf_matrix.shape[1]} features (terms)")

## Step 5: Apply SVD for Dimensionality Reduction

In [ ]:
# Determine optimal number of components
n_components = min(100, tfidf_matrix.shape[0] - 1, tfidf_matrix.shape[1] - 1)

print(f"\n🔧 Applying SVD with {n_components} components...")

# Initialize and fit SVD
svd = TruncatedSVD(n_components=n_components, random_state=42)
career_svd_matrix = svd.fit_transform(tfidf_matrix)

print(f"✅ SVD transformation complete!")
print(f"   - Original dimensions: {tfidf_matrix.shape}")
print(f"   - Reduced dimensions: {career_svd_matrix.shape}")
print(f"   - Explained variance ratio: {svd.explained_variance_ratio_.sum():.4f}")

## Step 6: Define Recommendation Function

In [ ]:
def recommend_careers_svd(user_profile, top_k=5):
    """
    Recommend careers based on user profile using SVD + Cosine Similarity
    
    Args:
        user_profile (dict): User's profile containing skills, interests, etc.
        top_k (int): Number of recommendations to return
    
    Returns:
        list: Top K recommended careers with scores
    """
    # Combine user profile into text
    user_text_parts = []
    user_text_parts.append(user_profile.get('department', ''))
    user_text_parts.append(user_profile.get('interests', ''))
    user_text_parts.append(' '.join(user_profile.get('skills', [])))
    user_text_parts.append(' '.join(user_profile.get('projects', [])))
    user_text_parts.append(' '.join(user_profile.get('certifications', [])))
    
    user_text = ' '.join([p for p in user_text_parts if p])
    
    print(f"\n🔍 User Profile Text:\n{user_text}\n")
    
    # Transform user text using TF-IDF and then SVD
    user_tfidf = tfidf.transform([user_text])
    user_svd = svd.transform(user_tfidf)
    
    # Calculate cosine similarity in reduced space
    similarities = cosine_similarity(user_svd, career_svd_matrix)[0]
    
    # Get top K indices
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    # Prepare results
    recommendations = []
    for idx in top_indices:
        career = careers_data[idx]
        recommendations.append({
            'career_name': career['career_name'],
            'category': career.get('category', 'N/A'),
            'similarity_score': float(similarities[idx]),
            'required_skills': career.get('required_skills', []),
            'recommended_courses': career.get('recommended_courses', []),
            'roadmap': career.get('roadmap', 'N/A')
        })
    
    return recommendations

print("✅ Recommendation function defined")

## Step 7: Get User Input

In [ ]:
# Collect user input
print("\n" + "="*60)
print("        WELCOME TO SVD CAREER RECOMMENDER SYSTEM")
print("="*60 + "\n")

name = input("Enter your name: ").strip()
department = input("Enter your department (e.g., CSE, ECE, MECH): ").strip()
skills_input = input("Enter your skills (comma-separated): ").strip()
interests = input("Enter your career interests: ").strip()
projects_input = input("Enter your projects (comma-separated): ").strip()
certifications_input = input("Enter your certifications (comma-separated): ").strip()

# Parse inputs
skills = [s.strip() for s in skills_input.split(',') if s.strip()]
projects = [p.strip() for p in projects_input.split(',') if p.strip()]
certifications = [c.strip() for c in certifications_input.split(',') if c.strip()]

# Create user profile
user_profile = {
    'name': name,
    'department': department,
    'skills': skills,
    'interests': interests,
    'projects': projects,
    'certifications': certifications
}

print("\n✅ Profile created successfully!")

## Step 8: Generate Recommendations

In [ ]:
# Get recommendations
print("\n" + "="*60)
print(f"  CAREER RECOMMENDATIONS FOR {user_profile['name'].upper()}")
print("="*60)

recommendations = recommend_careers_svd(user_profile, top_k=5)

# Display recommendations
print(f"\n🎯 Top 5 Career Recommendations (SVD Method):\n")

for i, rec in enumerate(recommendations, 1):
    print(f"\n{'='*60}")
    print(f"Rank #{i}: {rec['career_name']}")
    print(f"{'='*60}")
    print(f"📊 Similarity Score: {rec['similarity_score']:.4f}")
    print(f"📁 Category: {rec['category']}")
    print(f"\n💡 Required Skills:")
    for skill in rec['required_skills'][:5]:  # Show top 5 skills
        print(f"   • {skill}")
    print(f"\n📚 Recommended Courses:")
    for course in rec['recommended_courses'][:3]:  # Show top 3 courses
        print(f"   • {course}")
    print(f"\n🗺️ Career Roadmap:")
    print(f"   {rec['roadmap'][:150]}..." if len(rec['roadmap']) > 150 else f"   {rec['roadmap']}")

print("\n" + "="*60)
print("           RECOMMENDATIONS COMPLETE!")
print("="*60)

## Step 9: Visualize Results (Optional)

In [ ]:
import matplotlib.pyplot as plt

# Extract data for visualization
career_names = [rec['career_name'] for rec in recommendations]
scores = [rec['similarity_score'] for rec in recommendations]

# Create bar plot
plt.figure(figsize=(10, 6))
plt.barh(career_names, scores, color='coral')
plt.xlabel('Similarity Score', fontsize=12)
plt.ylabel('Career', fontsize=12)
plt.title(f'Top 5 Career Recommendations for {user_profile["name"]} (SVD)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Visualization complete!")

## Step 10: Analyze SVD Components (Optional)

In [ ]:
# Plot explained variance
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(svd.explained_variance_ratio_) + 1), 
         np.cumsum(svd.explained_variance_ratio_), 
         marker='o', linestyle='--', color='green')
plt.xlabel('Number of Components', fontsize=12)
plt.ylabel('Cumulative Explained Variance', fontsize=12)
plt.title('SVD Explained Variance', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"✅ Total variance explained by {n_components} components: {svd.explained_variance_ratio_.sum():.2%}")